In [1]:
pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.2 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import librosa
import scipy.signal as signal
import pandas as pd

# -----------------------------------------
# Utility Functions
# -----------------------------------------

def segment_audio(waveform, n_segments=3):
    L = len(waveform)
    split = np.array_split(waveform, n_segments)
    return split

def compute_slope(values):
    """Slope across early-mid-late = relaxation trend."""
    x = np.array([0, 1, 2])   # segment indices
    y = np.array(values)
    slope = np.polyfit(x, y, 1)[0]
    return slope

def hilbert_envelope(w):
    analytic = signal.hilbert(w)
    envelope = np.abs(analytic)
    return np.mean(envelope)

# -----------------------------------------
# Main TSRP + Calmness Metrics Function
# -----------------------------------------

def compute_TSRP_features(waveform, sr):

    segments = segment_audio(waveform, 3)

    # ----------- Base TSRP Features -----------
    rms_vals   = [librosa.feature.rms(y=s)[0].mean() for s in segments]
    zcr_vals   = [librosa.feature.zero_crossing_rate(y=s)[0].mean() for s in segments]

    rms_slope = compute_slope(rms_vals)
    zcr_slope = compute_slope(zcr_vals)

    # ----------- Additional Calmness Metrics -----------

    # 1. Temporal Smoothness Index (TSI) – low variation = calmer
    tsi_vals = [np.std(s) for s in segments]
    tsi_slope = compute_slope(tsi_vals)

    # 2. Spectral Stability Index (SSI) – centroid stability
    centroid_vals = [librosa.feature.spectral_centroid(y=s, sr=sr)[0].mean() for s in segments]
    ssi_slope = compute_slope(centroid_vals)

    # 3. Hilbert Envelope Decay Rate (HEDR)
    hilbert_vals = [hilbert_envelope(s) for s in segments]
    hedr_slope = compute_slope(hilbert_vals)

    return {
        # TSRP
        "RMS_Early": rms_vals[0], "RMS_Mid": rms_vals[1], "RMS_Late": rms_vals[2],
        "RMS_Slope": rms_slope,

        "ZCR_Early": zcr_vals[0], "ZCR_Mid": zcr_vals[1], "ZCR_Late": zcr_vals[2],
        "ZCR_Slope": zcr_slope,

        # New Calmness Indicators
        "TSI_Early": tsi_vals[0], "TSI_Mid": tsi_vals[1], "TSI_Late": tsi_vals[2],
        "TSI_Slope": tsi_slope,

        "SSI_Early": centroid_vals[0], "SSI_Mid": centroid_vals[1], "SSI_Late": centroid_vals[2],
        "SSI_Slope": ssi_slope,

        "HEDR_Early": hilbert_vals[0], "HEDR_Mid": hilbert_vals[1], "HEDR_Late": hilbert_vals[2],
        "HEDR_Slope": hedr_slope,
    }

# -----------------------------------------
# Apply to Full Dataset & Save to Excel
# -----------------------------------------

root_dir = "/kaggle/input/qmsat-dataset/ATS-data"
records = []

class_map = {"Music": 0, "Normal(Silence)": 1, "SpiritualMeditation": 2}

import os
for cls in os.listdir(root_dir):
    cls_path = os.path.join(root_dir, cls)
    if not os.path.isdir(cls_path): continue
    label = class_map.get(cls, None)
    if label is None: continue

    for file in os.listdir(cls_path):
        if file.endswith(".wav"):
            fp = os.path.join(cls_path, file)
            waveform, sr = librosa.load(fp, sr=16000)

            feats = compute_TSRP_features(waveform, sr)
            feats["Class"] = cls
            feats["File"] = file
            records.append(feats)

df = pd.DataFrame(records)

# Save to excel
df.to_excel("/kaggle/working/TSRP_and_Calmness_Features.xlsx", index=False)

df.head()


,RMS_Early,RMS_Mid,RMS_Late,RMS_Slope,ZCR_Early,ZCR_Mid,ZCR_Late,ZCR_Slope,TSI_Early,TSI_Mid,...,SSI_Early,SSI_Mid,SSI_Late,SSI_Slope,HEDR_Early,HEDR_Mid,HEDR_Late,HEDR_Slope,Class,File
0,0.023448,0.026114,0.026456,0.001504,0.261706,0.250831,0.250231,-0.005737,0.022935,0.026240,...,2711.132587,2630.485133,2630.135504,-40.498541,0.027277,0.030054,0.030662,0.001692,SpiritualMeditation,13_spiritual-meditation_f_51.wav
1,0.033430,0.023730,0.023942,-0.004744,0.310898,0.270889,0.282591,-0.014154,0.035128,0.023859,...,2700.725147,2348.050251,2467.780114,-116.472517,0.038037,0.027314,0.023357,-0.007340,SpiritualMeditation,12_spiritual-meditation_f_5.5.wav
2,0.126877,0.112316,0.091230,-0.017823,0.130628,0.134389,0.136286,0.002829,0.129095,0.114454,...,923.939379,1070.782353,1015.278697,45.669659,0.107213,0.101437,0.084448,-0.011383,SpiritualMeditation,19_spiritual-meditation_f_32.wav
3,0.010732,0.010856,0.008127,-0.001302,0.337055,0.331753,0.364511,0.013728,0.009831,0.009537,...,2271.654561,2311.371529,2475.345772,101.845606,0.011545,0.011567,0.008425,-0.001560,SpiritualMeditation,10_spiritual-meditation_f_51.wav
4,0.006472,0.049589,0.135607,0.064568,0.404027,0.288801,0.049929,-0.177049,0.005336,0.087198,...,2043.985723,1683.912764,560.745110,-741.620307,0.006431,0.053889,0.155270,0.074419,SpiritualMeditation,1_spiritual-meditation_f_23.wav
